# 08. Evaluación independiente

Se comparan el modelo base y el modelo ajustado utilizando la misma muestra boliviana de 100 publicaciones. Solo se evalúan filas con una etiqueta humana válida. La muestra no se utiliza para entrenar ni escoger hiperparámetros.


## A. Instalar y cargar la muestra

- **Qué hace:** abre la hoja que fue anotada sin observar predicciones.
- **Para qué:** construir la referencia humana de evaluación.
- **Por qué:** las métricas necesitan comparar predicciones con etiquetas independientes.


In [ ]:
%pip install -q "transformers==4.49.0" "pysentimiento==0.7.3" "emoji==2.16.0" scikit-learn matplotlib openpyxl


In [ ]:
from pathlib import Path

if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB")
else:
    RAIZ = Path.cwd()

if not (RAIZ / "TABLAS").is_dir():
    raise FileNotFoundError("Abre este notebook desde la carpeta COLAB o ajusta RAIZ.")
print("Carpeta de trabajo:", RAIZ)

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from pysentimiento.preprocessing import preprocess_tweet
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

PRUEBA = RAIZ / "TABLAS/05_preparacion/muestra_prueba_100_v4.xlsx"
MODELO_AJUSTADO = RAIZ / "outputs/modelos/robertuito_smps_v4/modelo_final"
SALIDA = RAIZ / "TABLAS/08_evaluacion"
SALIDA.mkdir(parents=True, exist_ok=True)

if not (MODELO_AJUSTADO / "config.json").is_file():
    raise FileNotFoundError("Ejecuta primero el cuaderno 07.")

muestra = pd.read_excel(PRUEBA, sheet_name="evaluacion", keep_default_na=False)
muestra["etiqueta_polaridad"] = muestra["etiqueta_polaridad"].astype(str).str.strip().str.lower()
permitidas = {"positiva", "negativa", "neutral"}
invalidas = set(muestra["etiqueta_polaridad"]) - permitidas - {""}
if invalidas:
    raise ValueError(f"Etiquetas no permitidas: {sorted(invalidas)}")

evaluables = muestra.loc[muestra["etiqueta_polaridad"].isin(permitidas)].copy()
if evaluables.empty:
    raise ValueError("Primero completa etiqueta_polaridad en la muestra del cuaderno 05.")
print("Filas evaluables:", len(evaluables), "/", len(muestra))
display(evaluables.head(10))


## B. Obtener predicciones comparables

- **Qué hace:** procesa cada publicación con el modelo base y con el ajustado usando la misma regla para textos largos.
- **Para qué:** comparar únicamente el efecto del ajuste.
- **Por qué:** cambiar simultáneamente el modelo y el procedimiento impediría atribuir la diferencia de resultados.


In [ ]:
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODELOS = {
    "base_TASS": "pysentimiento/robertuito-sentiment-analysis",
    "ajustado_SMPS": str(MODELO_AJUSTADO),
}
mapa = {0: "negativa", 1: "neutral", 2: "positiva"}

def cargar(ruta):
    tok = AutoTokenizer.from_pretrained(ruta)
    mod = AutoModelForSequenceClassification.from_pretrained(ruta).to(DISPOSITIVO).eval()
    return tok, mod

def predecir(texto, tok, mod):
    entrada = preprocess_tweet(str(texto), lang="es")
    lote = tok(
        entrada, max_length=128, truncation=True, stride=32,
        return_overflowing_tokens=True, padding=True, return_tensors="pt"
    )
    lote.pop("overflow_to_sample_mapping", None)
    lote = {k: v.to(DISPOSITIVO) for k, v in lote.items()}
    with torch.inference_mode():
        probabilidades = torch.softmax(mod(**lote).logits, dim=-1).mean(dim=0)
    return mapa[int(probabilidades.argmax())]

predicciones = evaluables[["id", "fuente", "etiqueta_polaridad"]].copy()
for nombre, ruta in MODELOS.items():
    tok, mod = cargar(ruta)
    predicciones[f"pred_{nombre}"] = [predecir(t, tok, mod) for t in evaluables["texto_procesado"]]
    del tok, mod
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

display(predicciones.head(100))


## C. Calcular métricas

- **Qué hace:** calcula métricas globales y por clase; además genera matrices de confusión con cantidades y porcentajes.
- **Para qué:** medir los aciertos generales y comprobar por separado el reconocimiento de cada polaridad.
- **Por qué:** accuracy sola puede ocultar un rendimiento deficiente en una clase minoritaria.


In [ ]:
clases = ["negativa", "neutral", "positiva"]
metricas_globales = []
reportes_clase = {}
matrices = {}

for nombre in MODELOS:
    reales = predicciones["etiqueta_polaridad"]
    estimadas = predicciones[f"pred_{nombre}"]
    reporte = classification_report(
        reales, estimadas, labels=clases, output_dict=True, zero_division=0
    )

    metricas_globales.append({
        "modelo": nombre,
        "n": int(len(reales)),
        "accuracy": float(accuracy_score(reales, estimadas)),
        "balanced_accuracy": float(balanced_accuracy_score(reales, estimadas)),
        "precision_macro": float(reporte["macro avg"]["precision"]),
        "recall_macro": float(reporte["macro avg"]["recall"]),
        "f1_macro": float(reporte["macro avg"]["f1-score"]),
        "f1_ponderado": float(reporte["weighted avg"]["f1-score"]),
    })

    por_clase = pd.DataFrame(reporte).T.loc[clases, ["precision", "recall", "f1-score", "support"]]
    por_clase.index.name = "clase"
    reportes_clase[nombre] = por_clase.reset_index()

    matriz = confusion_matrix(reales, estimadas, labels=clases)
    matriz_pct = confusion_matrix(reales, estimadas, labels=clases, normalize="true")
    matrices[nombre] = (matriz, matriz_pct)

    print(f"\nResultados por clase: {nombre}")
    display(por_clase.round(4))

    figura, ejes = plt.subplots(1, 2, figsize=(10, 4))
    for eje, datos_matriz, titulo, porcentaje in [
        (ejes[0], matriz, "Cantidad de publicaciones", False),
        (ejes[1], matriz_pct, "Porcentaje por clase real", True),
    ]:
        eje.imshow(datos_matriz, cmap="Blues", vmin=0, vmax=1 if porcentaje else None)
        eje.set_xticks(range(3), clases)
        eje.set_yticks(range(3), clases)
        eje.set_xlabel("Predicción del modelo")
        eje.set_ylabel("Etiqueta humana")
        eje.set_title(titulo)
        for i in range(3):
            for j in range(3):
                valor = f"{datos_matriz[i, j]:.1%}" if porcentaje else str(datos_matriz[i, j])
                eje.text(j, i, valor, ha="center", va="center")
    figura.suptitle(f"Matriz de confusión: {nombre}")
    figura.tight_layout()
    figura.savefig(SALIDA / f"matriz_{nombre}_v4.png", dpi=160, bbox_inches="tight")
    plt.show()

metricas = pd.DataFrame(metricas_globales)
print("Comparación global")
display(metricas.round(4))

predicciones.to_excel(SALIDA / "predicciones_prueba_v4.xlsx", index=False)
metricas.to_excel(SALIDA / "metricas_v4.xlsx", index=False)
with pd.ExcelWriter(SALIDA / "metricas_por_clase_v4.xlsx", engine="openpyxl") as escritor:
    for nombre, tabla in reportes_clase.items():
        tabla.to_excel(escritor, sheet_name=nombre, index=False)
with pd.ExcelWriter(SALIDA / "matrices_confusion_v4.xlsx", engine="openpyxl") as escritor:
    for nombre, (matriz, matriz_pct) in matrices.items():
        pd.DataFrame(matriz, index=clases, columns=clases).to_excel(escritor, sheet_name=f"{nombre}_cantidad")
        pd.DataFrame(matriz_pct, index=clases, columns=clases).to_excel(escritor, sheet_name=f"{nombre}_porcentaje")

columnas_grafico = ["accuracy", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_ponderado"]
fig, ax = plt.subplots(figsize=(11, 5))
metricas.set_index("modelo")[columnas_grafico].T.plot(
    kind="bar", ax=ax, color=["#2A6FBB", "#E07A2D"]
)
ax.set_title("Comparación de los modelos en la muestra boliviana")
ax.set_xlabel("Métrica")
ax.set_ylabel("Valor")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Modelo")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(SALIDA / "comparacion_metricas_v4.png", dpi=160, bbox_inches="tight")
plt.show()

print("Resultados guardados en:", SALIDA)
